<a href="https://colab.research.google.com/github/Institut-Castellbisbal/xatbot-2026-equip-2/blob/main/XatBot2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from IPython.display import display, HTML

codi_frontend = """
<div id="chat-box" style="width: 350px; height: 500px; border: 2px solid #2c3e50; border-radius: 10px; display: flex; flex-direction: column; background: #fdfdfd; font-family: sans-serif; margin: auto; box-shadow: 0 4px 15px rgba(0,0,0,0.2);">
    <div style="background: #2c3e50; color: white; padding: 12px; text-align: center; font-weight: bold; border-radius: 8px 8px 0 0;">
        Assistent LAN Party 2026
    </div>
    <div id="missatges" style="flex: 1; overflow-y: auto; padding: 15px; display: flex; flex-direction: column; gap: 10px; background: #f0f2f5;">
        <div style="background: #e3f2fd; color: #333; padding: 10px; border-radius: 10px; align-self: flex-start; font-size: 14px; border: 1px solid #bbdefb;">
            Hola, soc l'assistent de la LAN Party, en què et puc ajudar?
        </div>
    </div>
    <div style="display: flex; padding: 10px; border-top: 1px solid #ddd; background: #fff; border-radius: 0 0 10px 10px;">
        <input type="text" id="user-input" style="flex: 1; border: 1px solid #ccc; padding: 10px; border-radius: 20px; outline: none;" placeholder="Escriu un missatge...">
        <button onclick="enviar()" style="background: #007bff; color: white; border: none; margin-left: 8px; padding: 10px 18px; border-radius: 20px; cursor: pointer; font-weight: bold;">Enviar</button>
    </div>
</div>

<script>
    async function enviar() {
        const input = document.getElementById("user-input");
        const msg = input.value.trim();
        const box = document.getElementById("missatges");
        if (!msg) return;

        // 👇👇👇 POSA L'ENLLAÇ AQUÍ DINS 👇👇👇
        const urlBase = "https://denyse-unportrayed-viscerally.ngrok-free.dev/";
        // 👆👆👆 SENSE LA BARRA (/) AL FINAL 👆👆👆

        const uDiv = document.createElement("div");
        uDiv.style = "background: #007bff; color: white; padding: 10px; border-radius: 10px; align-self: flex-end; max-width: 80%; font-size: 14px;";
        uDiv.textContent = msg;
        box.appendChild(uDiv);
        input.value = "";
        box.scrollTop = box.scrollHeight;

        const tempDiv = document.createElement("div");
        tempDiv.style = "background: #e3f2fd; color: #333; padding: 10px; border-radius: 10px; align-self: flex-start; max-width: 80%; font-size: 14px; border: 1px solid #bbdefb;";
        tempDiv.textContent = "...";
        box.appendChild(tempDiv);
        box.scrollTop = box.scrollHeight;

        try {
            const resposta = await fetch(`${urlBase}/chat`, {
                method: "POST",
                headers: {
                    "Content-Type": "application/json",
                    "ngrok-skip-browser-warning": "true"
                },
                body: JSON.stringify({ missatge: msg })
            });
            const dades = await resposta.json();
            tempDiv.textContent = dades.resposta;
        } catch (e) {
            tempDiv.textContent = "🔴 Error: El servidor no respon. Revisa l'enllaç del codi.";
            tempDiv.style.background = "#fee2e2";
            tempDiv.style.border = "1px solid #fecaca";
        }
        box.scrollTop = box.scrollHeight;
    }
    document.getElementById("user-input").onkeypress = (e) => { if(e.key === 'Enter') enviar(); };
</script>
"""

display(HTML(codi_frontend))

In [4]:
# 1. INSTAL·LACIÓ DE LLIBRERIES
!pip install -q -U google-generativeai flask-cors pyngrok flask

import os
import time
import json
import threading
import google.generativeai as genai
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
from google.colab import userdata

# ---------------------------------------------------------
# 2. CREACIÓ DE LA BASE DE DADES (JSON)
# ---------------------------------------------------------
dades_lan_party = {
    "esdeveniment": "LAN Party Castellbisbal 2026",
    "ubicacio": "Els Costals, Avinguda de Pau Casals, 16, Castellbisbal",
    "dates": "Del 10 al 12 d'Abril de 2026",
    "connexio": {
        "xarxa": "LAN_CBAL_5G",
        "ip_servidor_jocs": "192.168.1.100",
        "velocitat": "10 Gbps simètrics"
    },
    "normativa": [
        "Prohibit portar menjar a la zona de PCs.",
        "Obligatori reciclar als contenidors EcoTech.",
        "Respectar les hores de descans a la zona de dormir."
    ],
    "competicions": ["League of Legends", "Valorant", "Mario Kart 8"]
}

with open("dades_lan.json", "w", encoding="utf-8") as fitxer:
    json.dump(dades_lan_party, fitxer, ensure_ascii=False, indent=4)

def carregar_dades_json(ruta_fitxer="dades_lan.json"):
    try:
        with open(ruta_fitxer, "r", encoding="utf-8") as fitxer:
            dades = json.load(fitxer)
            print("🟢 Base de dades carregada amb èxit!")
            return dades
    except Exception as e:
        print(f"🔴 ERROR: {e}")
        return None

memoria_bot = carregar_dades_json()

# ---------------------------------------------------------
# 3. CONFIGURACIÓ DE LA IA I VERIFICACIÓ
# ---------------------------------------------------------
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=api_key)

    print("Models disponibles per a la teva clau:")
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(m.name)

    model = genai.GenerativeModel('gemini-2.0-flash-lite')
except Exception as e:
    print(f"❌ Error amb la clau de Google: {e}")

# ---------------------------------------------------------
# 4. CREACIÓ DEL SERVIDOR (BACKEND)
# ---------------------------------------------------------
app = Flask(__name__)
CORS(app)

@app.route('/', methods=['GET'])
def inici():
    return "<h1 style='color: green;'>✅ EL SERVIDOR ESTÀ FUNCIONANT PERFECTAMENT!</h1>"

@app.route('/chat', methods=['POST'])
def chat():
    dades = request.json
    msg = dades.get("missatge", "")
    try:
        instruccions = """
        Ets el portaveu i assistent oficial de la LAN Party 2026.
        El teu to ha de ser educat i professional, adreçat a un públic jove pero serios, pero amb una mica de gracia.
        Amb respostes més curtes que llargues, sense saludar en cada missatge, només donant la informació i sense donar cap concell.
        En cas de que et preguntin qui ets o el primer missatge que envies abans de que la persona ho fagi (la teva presentació) ha de ser aquest: Hola, soc l'assistent de la LAN Party Castellbisbal 2026. En què et puc ajudar?
        Respon a cada pregunta with només la informació requerida, no et presentis a cada missatge, només en cas de que et preguntin "Qui ets?".
        NO DIGUIS RÉS MÉS APART DE LA INFORMACIÓ, NI CAP CONCELL, NI CAP FRASE BONICA, NOMÉS LA INFORMACIÓ!

        REGLES ESTRICTES:
        1. Està totalment prohibit utilitzar emojis o emoticones en les teves respostes.
        2. No t'inventis absolutament cap dada.
        3. Si et pregunten una cosa que no està a la informació de sota, respon educadament que no disposes d'aquesta informació en aquest moment.

        INFORMACIÓ OFICIAL DE LA LAN PARTY 2026:
        - Dates: "Del 10 d'abril a les 18h fins al 12 d'abril a les 18h."
        - Ubicació: "Els Costals, Avinguda de Pau Casals, 16, Castellbisbal"
        - Activitats i Tornejos: "Aquesta informació encara no esta disponible"
        - Preu i Entradas: "Aquesta informació encara no esta disponible"
        """

        prompt_final = f"{instruccions}\n\nPregunta de l'usuari: {msg}\nResposta formal:"
        res = model.generate_content(prompt_final)
        return jsonify({"resposta": res.text})

    except Exception as e:
        return jsonify({"resposta": f"Error IA: {str(e)}"})

# ---------------------------------------------------------
# 5. NETEJA, TÚNEL NGROK I EXECUCIÓ DEL SERVIDOR
# ---------------------------------------------------------
import os
import time
from google.colab import userdata

# Netegem processos anteriors
os.system("pkill -f ngrok")
try:
    ngrok.kill()
except:
    pass
time.sleep(2)

# Configuració del port
port_servidor = 8000

# SEGURETAT PRO+: Fem servir secrets per al token de ngrok
from google.colab import userdata

# Recuperem el token de forma segura
try:
    token_ngrok = userdata.get('token_ngrok')
    ngrok.set_auth_token(token_ngrok)
    print("✅ Token de ngrok carregat des dels Secrets.")
except userdata.SecretNotFoundError:
    print("❌ Error: No s'ha trobat el secret 'NGROK_TOKEN' a la configuració del Colab.")

# Connexió del túnel
try:
    public_url = ngrok.connect(port_servidor).public_url
    print("\n" + "="*60)
    print("🔗 COPIA AQUESTA NOVA URL PER AL FRONTEND:")
    print(f"👉 {public_url}")
    print("="*60)
    print("⚠️ ATENCIÓ: No paris aquesta cel·la o el xatbot deixarà de funcionar.\n")
except Exception as e:
    print(f"❌ Error al connectar ngrok: {e}")

# EXECUCIÓ DIRECTA (Sense threading per evitar que es tanqui)
if __name__ == '__main__':
    # use_reloader=False és necessari a Google Colab
    app.run(port=port_servidor, debug=False, use_reloader=False)

🟢 Base de dades carregada amb èxit!
Models disponibles per a la teva clau:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8000
INFO:werkzeug:Press CTRL+C to quit
